In [5]:
!git clone --depth 1 https://github.com/Digital-Pushkin-Lab/RuAdapt  # Скачиваем репозиторий RuAdapt в Colab
!git -C RuAdapt rev-parse HEAD  # Печатаем хэш коммита: его впишем в карточку датасета
!date +%d.%m.%Y  # Печатаем сегодняшнюю дату
!ls RuAdapt RuAdapt/Adapted_literature RuAdapt/Encyclopedic RuAdapt/Fairytales

fatal: destination path 'RuAdapt' already exists and is not an empty directory.
e124ea78ffae7ba105c4f48c2a21c6f8e5f80dd6
07.11.2026
RuAdapt:
Adapted_literature  Encyclopedic  Fairytales  README.md

RuAdapt/Adapted_literature:
levels.png
readme.md
zlatoust_paragraph_aligned_with_bleualign.csv
zlatoust_paragraph_aligned_with_CATS.csv
zlatoust_sentence_aligned_with_CATS.csv

RuAdapt/Encyclopedic:
lsslovar_B_to_A_par.csv   lsslovar_C_to_A_sent.csv  readme.md
lsslovar_B_to_A_sent.csv  lsslovar_C_to_B_par.csv
lsslovar_C_to_A_par.csv   lsslovar_C_to_B_sent.csv

RuAdapt/Fairytales:
df_fairytales_par.csv  df_fairytales_sent.csv  levels.png  readme.md


# **Этап 2. Загрузка трёх подкорпусов в одну таблицу**

In [6]:
import pandas as pd  # Библиотека для работы с таблицами

BASE = "RuAdapt"  # Папка, куда скачали RuAdapt

parts = []  # Сюда будем складывать куски таблицы из разных файлов

# --- 1. Адаптированная литература: берём только адаптированный текст (target) ---
d = pd.read_csv(f"{BASE}/Adapted_literature/zlatoust_paragraph_aligned_with_CATS.csv")
parts.append(pd.DataFrame({
    "text": d["target"], "level_raw": d["level"].astype(str),  # Текст и исходная метка уровня
    "subcorpus": "literature", "role": "target",  # Из какого подкорпуса и какая это сторона пары
    "pair_id": "lit_" + d.index.astype(str),  # Номер пары, чтобы потом не терять связь с оригиналом
    "cos_sim": d["cos_sim"], "file": "zlatoust_paragraph_CATS"}))  # Качество выравнивания и файл

# --- 2. Энциклопедические статьи: уровень задаёт имя файла; берём обе стороны пары ---
for name, (lvl_src, lvl_tgt) in {"B_to_A": ("B", "A"), "C_to_A": ("C", "A"), "C_to_B": ("C", "B")}.items():  # Для каждого файла: уровень оригинала и уровень упрощения
    d = pd.read_csv(f"{BASE}/Encyclopedic/lsslovar_{name}_par.csv")  # Читаем файл
    pid = f"enc_{name}_" + d.index.astype(str)  # Номер пары
    for col, lvl, role in [("source", lvl_src, "source"), ("target", lvl_tgt, "target")]:  # Две стороны пары
        parts.append(pd.DataFrame({
            "text": d[col], "level_raw": lvl, "subcorpus": "encyclopedic", "role": role,
            "pair_id": pid, "cos_sim": d["cos_sim"], "file": f"lsslovar_{name}_par"}))

# --- 3. Сказки: берём только адаптированный текст (target) ---
d = pd.read_csv(f"{BASE}/Fairytales/df_fairytales_par.csv")  # Читаем файл
parts.append(pd.DataFrame({
    "text": d["target"], "level_raw": d["level"].astype(str),
    "subcorpus": "fairytales", "role": "target",
    "pair_id": "fairy_" + d.index.astype(str),
    "cos_sim": d["cos_sim"], "file": "df_fairytales_par"}))

df = pd.concat(parts, ignore_index=True)  # Склеиваем все куски в одну таблицу
df["text"] = df["text"].astype(str)  # На всякий случай делаем все тексты строками

print("Всего строк в общей таблице:", len(df))  # Общий размер
print()
print("Строк по подкорпусам и ролям:")
print(df.groupby(["subcorpus", "role"]).size())  # Сколько строк в каждом подкорпусе
print()
print("Исходные метки уровней по подкорпусам:")
print(df.groupby(["subcorpus", "level_raw"]).size())  # Какие метки уровней есть
print()
print("Пропуски (NaN) по столбцам:")
print(df.isna().sum())  # Проверяем, нет ли пустых значений

Всего строк в общей таблице: 14782

Строк по подкорпусам и ролям:
subcorpus     role  
encyclopedic  source    3517
              target    3517
fairytales    target     134
literature    target    7614
dtype: int64

Исходные метки уровней по подкорпусам:
subcorpus     level_raw
encyclopedic  A            1017
              B            3008
              C            3009
fairytales    A1             12
              A1, A2          1
              A2, B1         75
              B1             33
              B1, B2         13
literature    a2            165
              a2_b1         217
              b1           4638
              b1_b2          32
              b1_c1        1459
              b2           1103
dtype: int64

Пропуски (NaN) по столбцам:
text         0
level_raw    0
subcorpus    0
role         0
pair_id      0
cos_sim      0
file         0
dtype: int64


# **Этап 3. Метки A/B/C, очистка текста, удаление мусора**

In [7]:
import re, unicodedata  # re: поиск по шаблонам в тексте; unicodedata: единая форма записи символов

n0 = len(df)  # Запоминаем размер таблицы до очистки

# --- 1. Приводим метки уровней к классам A / B / C ---
def to_class(raw):  # Функция: из исходной метки делает класс или None, если диапазон пересекает границу классов
    letters = set(re.findall(r"[abc]", raw.lower()))  # Достаём все буквы уровней (a1, b2 -> a, b)
    return letters.pop().upper() if len(letters) == 1 else None  # Одна буква = один класс; несколько = диапазон между классами

df["label"] = df["level_raw"].map(to_class)  # Новый столбец с классом
cross = df["label"].isna()  # Строки с диапазоном через границу классов
print("Убрано: диапазон уровней пересекает границу классов:", int(cross.sum()))
print(df[cross].groupby(["subcorpus", "level_raw"]).size())  # Что именно убрано
df = df[~cross].copy()  # Оставляем остальное

# --- 2. Чистим текст: пробелы, кавычки, тире ---
def clean(t):  # Функция очистки одного текста
    t = unicodedata.normalize("NFKC", t)  # Единая форма записи символов (в т.ч. неразрывный пробел -> обычный)
    t = "".join(c for c in unicodedata.normalize("NFD", t) if c not in "\u0300\u0301")  # Убираем знаки ударения (у части слов они есть)
    t = unicodedata.normalize("NFC", t)  # Собираем буквы обратно (й, ё и т.п.)
    t = re.sub(r"[\u2010\u2011\u2012\u2013\u2212]", "-", t)  # Разные дефисы и минусы (в т.ч. короткое тире) -> обычный дефис
    t = re.sub(r"[\u201c\u201d\u201e]", '"', t)  # Кавычки-«лапки» -> обычные прямые
    t = re.sub(r"\s+", " ", t)  # Любые повторы пробелов, переносы строк, табы -> один пробел
    return t.strip()  # Убираем пробелы по краям

df["text"] = df["text"].map(clean)  # Применяем очистку ко всем текстам

# --- 3. Убираем мусорные строки ---
words = df["text"].str.split().str.len()  # Число слов в каждой строке
no_cyr = ~df["text"].str.contains(r"[А-Яа-яЁё]")  # Строки вообще без русских букв
short = words < 3  # Слишком короткие (меньше 3 слов)
pause = df["text"].str.fullmatch(r"(Пауза\.?|\.+|\W+)")  # Строки-заглушки вроде «Пауза.» или только знаки

for name, mask in [("нет кириллицы", no_cyr), ("меньше 3 слов", short), ("«Пауза.»/только знаки", pause)]:  # Считаем по причинам
    print(f"Попало под правило «{name}»:", int(mask.sum()))
bad = no_cyr | short | pause  # Объединяем все правила
print("Всего убираем мусорных строк:", int(bad.sum()))
df = df[~bad].copy()  # Оставляем чистое
df["n_words"] = df["text"].str.split().str.len()  # Считаем длину в словах

print()
print("Было строк:", n0, "| осталось:", len(df))
print()
print(df.groupby(["subcorpus", "role", "label"]).size())  # Итог по классам
print()
print("Длина в словах (описание):")
print(df["n_words"].describe().round(1))  # Статистика длины

Убрано: диапазон уровней пересекает границу классов: 1751
subcorpus   level_raw
fairytales  A2, B1         75
literature  a2_b1         217
            b1_c1        1459
dtype: int64
Попало под правило «нет кириллицы»: 0
Попало под правило «меньше 3 слов»: 184
Попало под правило ««Пауза.»/только знаки»: 56
Всего убираем мусорных строк: 184

Было строк: 14782 | осталось: 12847

subcorpus     role    label
encyclopedic  source  B         508
                      C        3001
              target  A        1013
                      B        2492
fairytales    target  A          13
                      B          45
literature    target  A         163
                      B        5612
dtype: int64

Длина в словах (описание):
count    12847.0
mean        35.8
std         33.7
min          3.0
25%         13.0
50%         25.0
75%         46.0
max        389.0
Name: n_words, dtype: float64


# **Этап 4. Дедупликация и конфликты уровней.**

In [8]:
n_before = len(df)  # Размер до удаления дубликатов

# --- 1. Ключ для сравнения: текст в нижнем регистре без знаков препинания и пробелов ---
df["key"] = df["text"].str.lower().str.replace(r"[^а-яёa-z0-9]", "", regex=True)  # Так «Он пришёл.» и «он пришёл» считаются одним текстом

# --- 2. Тексты, которые встречаются с РАЗНЫМИ уровнями (конфликт) ---
n_labels = df.groupby("key")["label"].nunique()  # Сколько разных классов у каждого текста
conflict_keys = n_labels[n_labels > 1].index  # Тексты с конфликтом: непонятно, какой у них уровень
conf = df["key"].isin(conflict_keys)  # Строки с такими текстами
print("Строк с конфликтом уровней (текст встречается с разными классами):", int(conf.sum()))
print("Из них уникальных текстов:", len(conflict_keys))
print(df[conf].groupby(["subcorpus", "role", "label"]).size())  # Откуда они
df = df[~conf].copy()  # Конфликтные тексты убираем полностью

# --- 3. Точные дубликаты с одним уровнем: оставляем строку с наибольшим cos_sim ---
df = df.sort_values("cos_sim", ascending=False)  # Сначала самые качественные пары
dup = df.duplicated("key", keep="first")  # Все повторы после первой (лучшей) строки
print()
print("Точных дубликатов с тем же уровнем удалено:", int(dup.sum()))
print(df[dup].groupby(["subcorpus", "role"]).size())  # Откуда дубликаты
df = df[~dup].sort_index().copy()  # Убираем повторы и возвращаем исходный порядок

# --- 4. Итоги ---
n_after = len(df)  # Размер после
print()
print("Объём ДО:", n_before, "| ПОСЛЕ:", n_after)
print("Доля удалённого на этом этапе:", round(100 * (n_before - n_after) / n_before, 1), "%")
print("Доля удалённого от самого начала (14782):", round(100 * (14782 - n_after) / 14782, 1), "%")
print()
print(df.groupby(["subcorpus", "role", "label"]).size())  # Итоговые объёмы
print()
print("Итого по классам:")
print(df["label"].value_counts())  # Баланс классов

Строк с конфликтом уровней (текст встречается с разными классами): 3668
Из них уникальных текстов: 1560
subcorpus     role    label
encyclopedic  source  B         237
                      C        1766
              target  A         145
                      B        1520
dtype: int64

Точных дубликатов с тем же уровнем удалено: 1085
subcorpus     role  
encyclopedic  source    596
              target    482
fairytales    target      2
literature    target      5
dtype: int64

Объём ДО: 12847 | ПОСЛЕ: 8094
Доля удалённого на этом этапе: 37.0 %
Доля удалённого от самого начала (14782): 45.2 %

subcorpus     role    label
encyclopedic  source  B          23
                      C         887
              target  A         433
                      B         925
fairytales    target  A          11
                      B          45
literature    target  A         163
                      B        5607
dtype: int64

Итого по классам:
label
B    6600
C     887
A     607
Name: count,

# **Этап 5. Примеры, деление на обучение/тест, проверка утечек**

In [9]:
import numpy as np  # Числа и массивы
from sklearn.model_selection import StratifiedGroupKFold  # Деление на части с учётом групп и классов
from sklearn.feature_extraction.text import TfidfVectorizer  # Превращает тексты в числа (для поиска почти-дубликатов)
from sklearn.metrics.pairwise import cosine_similarity  # Косинусная близость между текстами

# --- 1. Читаем глазами 10 случайных примеров ---
sample = df.sample(10, random_state=42)  # 10 случайных строк (random_state=42 -> каждый раз одни и те же)
for i, (_, r) in enumerate(sample.iterrows(), 1):  # Печатаем по одному
    print(f"{i}. [{r['subcorpus']} | {r['role']} | класс {r['label']} | исходно {r['level_raw']}]")  # Откуда и какой уровень
    print("   ", r["text"][:300])  # Первые 300 символов текста
print()

# --- 2. Делим на обучающую (80%) и тестовую (20%) части ---
df = df.reset_index(drop=True)  # Нумеруем строки заново
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)  # 5 частей: одна станет тестом (20%)
train_idx, test_idx = next(sgkf.split(df, df["label"], groups=df["pair_id"]))  # Обе стороны одной пары не разделяются
df["split"] = "train"  # Всем сначала «обучающая»
df.loc[test_idx, "split"] = "test"  # Выбранным - «тестовая»

print("Размер частей:")
print(df["split"].value_counts())
print()
print("Классы в частях (строк):")
print(pd.crosstab(df["split"], df["label"]))
print()
print("Классы в частях (доля, %):")
print((pd.crosstab(df["split"], df["label"], normalize="index") * 100).round(1))
print()
print("Подкорпуса в частях:")
print(pd.crosstab(df["split"], df["subcorpus"]))
print()

# --- 3. Проверки на утечку ---
tr, te = df[df["split"] == "train"], df[df["split"] == "test"]  # Две части отдельно
print("Проверка 1. Общих номеров пар (групп) между train и test:", len(set(tr["pair_id"]) & set(te["pair_id"])))  # Должно быть 0
print("Проверка 2. Общих нормализованных текстов между train и test:", len(set(tr["key"]) & set(te["key"])))  # Должно быть 0

vec = TfidfVectorizer().fit(df["text"])  # Строим словарь по всем текстам
sim = cosine_similarity(vec.transform(te["text"]), vec.transform(tr["text"]))  # Близость каждого теста к каждому train-тексту
max_sim = sim.max(axis=1)  # Для каждого тестового текста — самый похожий в train
print("Проверка 3. Тестовых текстов, у которых в train есть почти-копия (TF-IDF > 0.9):", int((max_sim > 0.9).sum()), "из", len(te))
print("   Доля, %:", round(100 * (max_sim > 0.9).mean(), 1))

1. [literature | target | класс B | исходно b1]
    Сергей больше не ездил к графу. Не хотел. Об Ольге не думал и думать не хотел. Много работал. Конечно, иногда он вспоминал лесной домик, где жила «девушка в красном», дорогу в Те нево, свидание в пещере... Он чувствовал боль. И теперь, когда покончил с Ольгой, он уже смотрел на эту «поэзию» не так, 
2. [literature | target | класс B | исходно b1]
    — Она... здесь, — медленно проговорил Рогожин. — Где же? Рогожин поднял глаза на князя и внимательно посмотрелна него: — Пойдём... Он всё говорил шёпотом и не торопясь, медленно, задумчиво. Вошли в кабинет. В комнате было очень темно. — Ты бы свечку зажёг? — сказал князь. — Нет, не надо, — ответил Р
3. [literature | target | класс B | исходно b1]
    — Зачем ты это прибавил? И вот опять разозлился, — сказал князь.
4. [literature | target | класс B | исходно b1]
    — Как вас здесь много! Наверное, весело! Здравствуй, Алексис! — сказала она графу.
5. [encyclopedic | target | класс B | исхо

# **Этап 6. Итоговая статистика и сохранение набора**

In [11]:
# --- 1. Итоговое распределение по классам и подкорпусам (для карточки, раздел «Распределение») ---
tab = pd.crosstab(df["label"], df["subcorpus"], margins=True, margins_name="Всего")  # Таблица: классы x подкорпуса
print("Единиц по классам и подкорпусам:")
print(tab)
print()
print("Доля классов, %:")
print((df["label"].value_counts(normalize=True).sort_index() * 100).round(1))  # Доли классов
print()

# --- 2. Какие исходные уровни остались (до объединения в A/B/C) ---
print("Исходные уровни, которые вошли в итоговый набор:")
print(df.groupby(["subcorpus", "role", "level_raw"]).size())
print()

# --- 3. Длина текстов по классам (слов в абзаце) ---
print("Длина абзаца в словах по классам:")
print(df.groupby("label")["n_words"].describe(percentiles=[0.05, 0.5, 0.95]).round(1))
print()

# --- 4. Итоговая таблица для раздела «Разделение» ---
sp = df["split"].value_counts()  # Сколько единиц в train и test
for part in ["train", "test"]:  # Печатаем строки в формате карточки
    print(f"{part}: {sp[part]} единиц, {round(100 * sp[part] / len(df), 1)} %")
print()

# --- 5. Сохраняем нормализованный набор в файл ---
out = df[["text", "label", "level_raw", "subcorpus", "role", "pair_id", "cos_sim", "n_words", "split"]]  # Нужные столбцы
out.to_csv("ruadapt_normalized.csv", index=False, encoding="utf-8")  # Пишем CSV
print("Сохранено строк:", len(out), "-> файл ruadapt_normalized.csv")

Единиц по классам и подкорпусам:
subcorpus  encyclopedic  fairytales  literature  Всего
label                                                 
A                   433          11         163    607
B                   948          45        5607   6600
C                   887           0           0    887
Всего              2268          56        5770   8094

Доля классов, %:
label
A     7.5
B    81.5
C    11.0
Name: proportion, dtype: float64

Исходные уровни, которые вошли в итоговый набор:
subcorpus     role    level_raw
encyclopedic  source  B              23
                      C             887
              target  A             433
                      B             925
fairytales    target  A1             10
                      A1, A2          1
                      B1             32
                      B1, B2         13
literature    target  a2            163
                      b1           4615
                      b1_b2          31
                      b2    